# **NN Prediction Model**

---
## **1. Big Picture and Objectives**

### **Objective**
By June 2026 (end of the Premier League season), develop and evaluate a feedforward (Sequential) neural network that predicts Premier League match outcomes (home win, draw, away win) using historical FBref data scraped from FBref, achieving a **Macro F1-score of at least 0.70** on a held-out test season, while applying the concepts learned in the 'Neural Networks and Deep Learning' course by deeplearning.ai (forward/backward propagation, gradient descent, regularization, and hyperparameter tuning).

> **Why Macro F1 over Accuracy?** The dataset is imbalanced (Home Win ~45%, Away Win ~30%, Draw ~25%). A naive model predicting "Home Win" always achieves ~45% accuracy without learning anything. Macro F1 weights all three classes equally, ensuring the model genuinely learns to predict draws and away wins — not just the dominant class.

### **Similar Projects**
The next are good examples of similar projects that have applied neural networks and machine learning techniques to predict football match outcomes:



- **[lorenzopalaia/Football-Prediction](https://github.com/lorenzopalaia/Football-Prediction)** (Palaia, 2024): Keras NN on historical data; full notebooks for prep/training/eval.
- **[giovannicampa/football_match_results_prediction](https://github.com/giovannicampa/football_match_results_prediction)** (Campa, 2019): MLP/TensorFlow predict goals/outcomes; 55–59% accuracy.
- **[AndrewCarterUK/football-predictor](https://github.com/AndrewCarterUK/football-predictor)** (Carter, 2018): Deep NN for Premier League using team stats.
- **[yuliang419/football-predictor](https://github.com/yuliang419/football-predictor)** (Yuliang, 2022): Simple NN for EPL from recent performance.

### **Potential Pipeline**
Get data (FBref historical match data) → Preprocess (clean, feature engineering) → Train/Test split → Build NN model (Keras Sequential) → Train (forward/backward prop, gradient descent) → Evaluate (Macro F1-score on test set) → Hyperparameter tuning → Final evaluation.


---
## **2. Get the Data**

In [1]:
# read all the csv files inside the data folder to put in one dataframe

from pathlib import Path

import pandas as pd

cwd = Path.cwd()
path = cwd / "src" / "data"

if not path.exists():
    path = cwd.parent / "data"

if not path.exists():
    raise FileNotFoundError(f"Could not find data directory from {cwd}")

df = pd.DataFrame()
for file in path.iterdir():
    if file.suffix == ".csv":
        df = pd.concat([df, pd.read_csv(file)], ignore_index=True)

df

,week,day,date,time,home,score,away,attendance,venue,referee,match_report
0,1.0,Sat,2015-08-08,12:45(06:45),Manchester Utd,1–0,Tottenham Hotspur,75261.0,Old Trafford,Jonathan Moss,https://fbref.com/en/matches/86cdfeba/Manchest...
1,1.0,Sat,2015-08-08,15:00(09:00),Leicester City,4–2,Sunderland,32242.0,King Power Stadium,Lee Mason,https://fbref.com/en/matches/a6cda14d/Leiceste...
2,1.0,Sat,2015-08-08,15:00(09:00),Bournemouth,0–1,Aston Villa,11155.0,Vitality Stadium,Mark Clattenburg,https://fbref.com/en/matches/df747efb/Bournemo...
3,1.0,Sat,2015-08-08,15:00(09:00),Everton,2–2,Watford,39063.0,Goodison Park,Mike Jones,https://fbref.com/en/matches/ac0bb534/Everton-...
4,1.0,Sat,2015-08-08,15:00(09:00),Norwich City,1–3,Crystal Palace,27036.0,Carrow Road,Simon Hooper,https://fbref.com/en/matches/47257ad7/Norwich-...
...,...,...,...,...,...,...,...,...,...,...,...
4691,38.0,Sun,2026-05-24,16:00(10:00),Liverpool,NaN,Brentford,NaN,Anfield,NaN,https://fbref.com/en/stathead/matchup/teams/cd...
4692,38.0,Sun,2026-05-24,16:00(10:00),Burnley,NaN,Wolves,NaN,Turf Moor,NaN,https://fbref.com/en/stathead/matchup/teams/94...
4693,38.0,Sun,2026-05-24,16:00(10:00),West Ham United,NaN,Leeds United,NaN,London Stadium,NaN,https://fbref.com/en/stathead/matchup/teams/7c...
4694,38.0,Sun,2026-05-24,16:00(10:00),Manchester City,NaN,Aston Villa,NaN,Etihad Stadium,NaN,https://fbref.com/en/stathead/matchup/teams/b8...


---
## **3. Exploratory Analysis and Insights**

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4696 entries, 0 to 4695
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   week          4180 non-null   float64
 1   day           4180 non-null   str    
 2   date          4180 non-null   str    
 3   time          4180 non-null   str    
 4   home          4180 non-null   str    
 5   score         4101 non-null   str    
 6   away          4180 non-null   str    
 7   attendance    3660 non-null   float64
 8   venue         4180 non-null   str    
 9   referee       4101 non-null   str    
 10  match_report  4179 non-null   str    
dtypes: float64(2), str(9)
memory usage: 403.7 KB


In [3]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
week,4180.0,19.500000,10.967168,1.0,10.00,19.5,29.00,38.0
attendance,3660.0,38483.679508,16802.489603,2000.0,25532.75,32233.5,52986.75,83222.0
